# Setup

In [1]:
import sys, os
# torch/triton/pandas уже есть в Kaggle GPU-образе; путь к встроенному src
sys.path.insert(0, '/kaggle/working')

In [2]:
import torch
import gc
import triton
import pandas as pd

from src.backend.flash_attention import (
    torch_attention,
    flash_attn_forward,
    flash_attn_backward,
    FlashAttentionFunc,
    FlashAttention,
)

# 3.1 Forward FlashAttention

In [3]:
B, H, N, D = 2, 4, 256, 64
Q = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)
K = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)
V = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)

ref = torch_attention(Q, K, V, causal=True)
out, lse = flash_attn_forward(Q, K, V, causal=True)

torch.testing.assert_close(out, ref, atol=1e-2, rtol=1e-2)
print('Forward test passed!')

Forward test passed!


# 3.2 Backward FlashAttention

In [4]:
B, H, N, D = 2, 4, 128, 64

Q = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16, requires_grad=True)
K = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16, requires_grad=True)
V = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16, requires_grad=True)

Q_ref = Q.detach().clone().requires_grad_(True)
K_ref = K.detach().clone().requires_grad_(True)
V_ref = V.detach().clone().requires_grad_(True)

ref_out = torch_attention(Q_ref, K_ref, V_ref, causal=True)
ref_out.sum().backward()

module = FlashAttention(causal=True)
flash_out = module(Q, K, V)
flash_out.sum().backward()

torch.testing.assert_close(flash_out, ref_out, atol=1e-2, rtol=1e-2)
torch.testing.assert_close(Q.grad, Q_ref.grad, atol=2e-2, rtol=2e-2)
torch.testing.assert_close(K.grad, K_ref.grad, atol=2e-2, rtol=2e-2)
torch.testing.assert_close(V.grad, V_ref.grad, atol=2e-2, rtol=2e-2)
print('Backward test passed!')

Backward test passed!


# 3.3 Benchmarks

In [5]:
def bench_torch_fwd(Q, K, V):
    return torch_attention(Q, K, V, causal=True)

def bench_flash_fwd(Q, K, V):
    return flash_attn_forward(Q, K, V, causal=True)

def bench_torch_fwd_bwd(Q, K, V):
    o = torch_attention(Q, K, V, causal=True)
    o.sum().backward()

def bench_flash_fwd_bwd(Q, K, V):
    o = FlashAttentionFunc.apply(Q, K, V, True)
    o.sum().backward()

B, H, D = 4, 8, 64
SEQ_LENS = [128, 256, 512, 1024, 2048]
rows = []

for N in SEQ_LENS:
    def make_inputs(grad=False):
        q = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16, requires_grad=grad)
        k = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16, requires_grad=grad)
        v = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16, requires_grad=grad)
        return q, k, v

    # forward timing
    Q, K, V = make_inputs()
    t_torch_fwd = triton.testing.do_bench(lambda: bench_torch_fwd(Q, K, V))
    t_flash_fwd = triton.testing.do_bench(lambda: bench_flash_fwd(Q, K, V))

    # memory: torch
    del Q, K, V
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    q, k, v = make_inputs(grad=True)
    bench_torch_fwd_bwd(q, k, v)
    torch.cuda.synchronize()
    mem_torch = torch.cuda.max_memory_allocated() / 1024**2

    # memory: flash
    del q, k, v
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    q, k, v = make_inputs(grad=True)
    bench_flash_fwd_bwd(q, k, v)
    torch.cuda.synchronize()
    mem_flash = torch.cuda.max_memory_allocated() / 1024**2

    # fwd+bwd timing
    del q, k, v
    gc.collect()
    torch.cuda.empty_cache()
    q, k, v = make_inputs(grad=True)
    t_torch_total = triton.testing.do_bench(lambda: bench_torch_fwd_bwd(q, k, v))
    q, k, v = make_inputs(grad=True)
    t_flash_total = triton.testing.do_bench(lambda: bench_flash_fwd_bwd(q, k, v))

    rows.append({
        'N': N,
        'torch_fwd_ms': round(t_torch_fwd, 3),
        'flash_fwd_ms': round(t_flash_fwd, 3),
        'fwd_speedup': round(t_torch_fwd / t_flash_fwd, 2),
        'torch_total_ms': round(t_torch_total, 3),
        'flash_total_ms': round(t_flash_total, 3),
        'total_speedup': round(t_torch_total / t_flash_total, 2),
        'torch_mem_MB': round(mem_torch, 1),
        'flash_mem_MB': round(mem_flash, 1),
        'mem_ratio': round(mem_torch / mem_flash, 2),
    })
    print(f'N={N} done')

df = pd.DataFrame(rows)
print()
print(df.to_string(index=False))

N=128 done


N=256 done


N=512 done


N=1024 done


N=2048 done

   N  torch_fwd_ms  flash_fwd_ms  fwd_speedup  torch_total_ms  flash_total_ms  total_speedup  torch_mem_MB  flash_mem_MB  mem_ratio
 128         0.109         0.822         0.13           0.217           2.751           0.08          25.0          24.0       1.04
 256         0.192         2.342         0.08           0.596           8.473           0.07          39.6          29.5       1.34
 512         0.695         7.979         0.09           2.110          29.679           0.07          92.8          40.6       2.29
1024         2.688        29.586         0.09           7.844         110.298           0.07         295.5          62.6       4.72
2048        11.256       113.714         0.10          30.797         426.047           0.07        1086.5         106.8      10.18
